[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_8_robotics_projects/25_capstone_template.ipynb)

# Notebook 25 — Capstone Project Template

---

**Part 8 · Robotics Projects**

```
╔════════════════════════════════════════════════════════════════════════╗
║                                                                        ║
║   YOUR 6D POSE ESTIMATION PROJECT                                      ║
║                                                                        ║
║   This template guides you through designing and implementing your     ║
║   own robotics perception system using everything from this course.    ║
║                                                                        ║
╚════════════════════════════════════════════════════════════════════════╝
```

> **You've covered the full spectrum**: Camera model → Calibration → OpenCV → ArUco → Stereo Vision → Deep Learning 6D Pose. Now build something real. This template gives you the structure; you fill in the substance.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-contrib-python -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
print('All imports OK — let\'s build something great!')

## Section 0 — Project Brief

Fill this in before writing any code. Clarity here saves hours of confused implementation.

---

**Project Title**: *(e.g., "Autonomous bin-picking pose estimation for irregular parts")*

---

**Problem Statement**

*(What real-world problem does this solve? Be specific.)*

---

**Object(s) to localize**

*(What objects? Do you have a CAD model? Are they textured or textureless?)*

---

**Input data**

- [ ] USB webcam (RGB only)
- [ ] Intel RealSense (RGB-D)
- [ ] ZED / Stereo camera
- [ ] Pre-recorded video/images
- [ ] Synthetic data (generated)

---

**Output / Action**

*(What happens with the pose? Robot command? AR overlay? Grasp planning?)*

---

**Constraints**

- Real-time required? Yes / No — target: ___ FPS
- GPU available? Yes / No
- Markers allowed? Yes / No
- Training data available? Yes / No

## Section 1 — Method Selection

Use this decision tree to pick your approach:

```
Q1: Can you place markers on or near the object?
 YES → ArUco + solvePnP (NB 09–15)          → Go to Step 1A
 NO  → Continue to Q2

Q2: Do you have RGB-D (depth camera)?
 NO  → MegaPose + ViSP (NB 22)               → Go to Step 1B
 YES → Continue to Q3

Q3: Is there a fixed set of objects (no new ones)?
 YES + training data → EfficientPose (NB 20)  → Go to Step 1C
 NO (new objects arrive) → Continue to Q4

Q4: Do you have a CAD model?
 YES → FreeZe or FoundationPose (NB 21)       → Go to Step 1D
 NO  → Need to create one (Blender/scan) then use 1D
```

---

**My chosen method**: *(write here)*

**Reason**: *(why this method fits your constraints)*

In [ ]:
# ── Project Configuration ─────────────────────────────────────────────────
# Fill in your project parameters here.

PROJECT_NAME = 'My 6D Pose Project'   # Change this

# ── Camera settings ───────────────────────────────────────────────────────
CAMERA_INDEX = 0       # 0 = default camera, 1 = second camera, etc.
FRAME_WIDTH  = 640
FRAME_HEIGHT = 480

# ── Calibration ───────────────────────────────────────────────────────────
# Use your calibration file from NB 07, OR enter known values below
CALIB_FILE = '../assets/calibration/camera_calibration.npz'

def load_calibration(calib_file):
    """Load camera intrinsics from calibration file."""
    try:
        data = np.load(calib_file)
        K    = data['K']
        dist = data['dist']
        print(f'Loaded calibration from {calib_file}')
        return K, dist
    except FileNotFoundError:
        print(f'Calibration file not found: {calib_file}')
        print('Using fallback intrinsics (replace with your calibration!)')
        # Fallback: approximate values for a 640x480 webcam
        fx = fy = 600.0
        cx, cy = FRAME_WIDTH / 2, FRAME_HEIGHT / 2
        K    = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
        dist = np.zeros(5)
        return K, dist

K, DIST = load_calibration(CALIB_FILE)
print(f'\nK =\n{K}')
print(f'dist = {DIST}')

## Step 1A — ArUco + solvePnP Path

*(Complete this section if you chose the ArUco approach. Delete/skip if using DL methods.)*

**Planning questions:**
- Which marker dictionary? `DICT_4X4_50` (good default) vs `DICT_6X6_250` (higher capacity)
- What physical marker size will you print?
- Where will markers be placed relative to the object/station?

In [ ]:
# ── Step 1A: ArUco Setup ──────────────────────────────────────────────────

# ── ArUco API compatibility shim (copy from NB 12) ────────────────────────
def get_aruco_dict(dict_id=cv2.aruco.DICT_4X4_50):
    try:
        return cv2.aruco.getPredefinedDictionary(dict_id)
    except AttributeError:
        return cv2.aruco.Dictionary_get(dict_id)

def get_aruco_params():
    try:
        return cv2.aruco.DetectorParameters()
    except AttributeError:
        return cv2.aruco.DetectorParameters_create()

def detect_markers(gray, aruco_dict, params):
    try:
        detector = cv2.aruco.ArucoDetector(aruco_dict, params)
        return detector.detectMarkers(gray)
    except AttributeError:
        return cv2.aruco.detectMarkers(gray, aruco_dict, parameters=params)

# ── YOUR CONFIGURATION ────────────────────────────────────────────────────
MARKER_DICT_ID  = cv2.aruco.DICT_4X4_50   # TODO: choose your dictionary
MARKER_SIZE_M   = 0.10                     # TODO: your physical marker size (meters)
TARGET_IDS      = [0, 1, 2]               # TODO: which marker IDs to track

ARUCO_DICT   = get_aruco_dict(MARKER_DICT_ID)
ARUCO_PARAMS = get_aruco_params()

print(f'ArUco dictionary: DICT_4X4_50')
print(f'Marker size: {MARKER_SIZE_M*100:.0f} cm')
print(f'Tracking IDs: {TARGET_IDS}')

# ── YOUR DETECTION LOOP ───────────────────────────────────────────────────
# TODO: Replace this comment block with your detection + pose loop

# def run_aruco_pipeline(frame):
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     corners, ids, rejected = detect_markers(gray, ARUCO_DICT, ARUCO_PARAMS)
#
#     if ids is not None:
#         for marker_id in TARGET_IDS:
#             if marker_id in ids.flatten():
#                 idx = list(ids.flatten()).index(marker_id)
#                 rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
#                     corners[idx:idx+1], MARKER_SIZE_M, K, DIST)
#                 rvec, tvec = rvecs[0][0], tvecs[0][0]
#
#                 # YOUR LOGIC HERE: compute errors, generate commands, etc.
#                 process_pose(marker_id, rvec, tvec)
#
#     return frame

print('ArUco pipeline scaffold ready. Uncomment and complete the detection loop.')

## Step 1B — MegaPose + ViSP Path

*(Complete this section if your system has no depth camera and you need CAD-model-based pose. Delete/skip if using other methods.)*

**Requirements checklist:**
- [ ] CAD model prepared: `model.obj` + `model.mtl` + `texture.jpg` in one folder
- [ ] MegaPose server installed and running on GPU machine
- [ ] ViSP installed (`pip install visp` or from source)
- [ ] Camera calibration file ready

In [ ]:
# ── Step 1B: MegaPose + ViSP Scaffold ────────────────────────────────────

# YOUR MODEL PATH
MODEL_PATH = './my_object/'   # TODO: path to your model folder

# MegaPose server connection
MEGAPOSE_HOST = '127.0.0.1'   # TODO: server address
MEGAPOSE_PORT = 5555

# ── MegaPose initialization (run once) ────────────────────────────────────
# TODO: Replace with actual MegaPose client from happypose
# from happypose.toolbox.inference.types import ObservationTensor
# from happypose.toolbox.inference.pose_estimator import PoseEstimator

# estimator = PoseEstimator.from_pretrained('megapose-1.0-RGB')
# estimator.load_model(MODEL_PATH)

# ── Per-frame pipeline ────────────────────────────────────────────────────
# def run_megapose_pipeline(frame, bbox):
#     """
#     bbox: (x, y, w, h) from your object detector (YOLO, etc.)
#     """
#     obs = ObservationTensor.from_numpy(frame, K)
#     result = estimator.run_inference_pipeline(obs, detections=[bbox])
#     pose = result.poses[0]   # 4x4 matrix
#     R = pose[:3, :3]
#     t = pose[:3, 3]
#     return R, t

# ── ViSP tracking (after initialization) ─────────────────────────────────
# See: https://visp-doc.inria.fr/doxygen/visp-daily/tutorial-tracking-megapose.html

print('MegaPose + ViSP scaffold ready. See NB 22 for full setup instructions.')

---

> ### 💡 What does "Docker recommended" mean in the checklist below?
>
> If you chose the **EfficientPose path (Step 1C)**, the checklist mentions Docker. In short:
>
> **Docker** creates an isolated mini-environment on your machine so that EfficientPose's old TensorFlow 1.14 requirement doesn't conflict with anything else on your system.
>
> - Full plain-English explanation → see **NB 20, the "New to Docker?" section**
> - Install Docker Desktop → https://www.docker.com/products/docker-desktop
> - Setup commands → see **NB 20, Section 3 (Setting Up EfficientPose)**
>
> If you chose FreeZe or FoundationPose (Step 1D), Docker is optional — those paths offer a `conda` alternative that works on most machines.

---

## Step 1C — EfficientPose Path

*(If using EfficientPose. Delete/skip otherwise.)*

See NB 20 (`20_efficientpose.ipynb`) for the full inference pipeline and environment setup.

**Requirements checklist:**
- [ ] TF 1.14 + Python 3.7 environment (Docker recommended)
- [ ] Pretrained weights downloaded
- [ ] Either: use existing LineMOD class, OR retrain on your object

## Step 1D — FreeZe / FoundationPose Path

*(If using zero-shot foundation models. Delete/skip otherwise.)*

See NB 21 (`21_foundationpose_freeze.ipynb`) for architecture details and setup.

**Requirements checklist:**
- [ ] RGB-D camera set up with calibration
- [ ] CAD model: `model.obj` + point cloud
- [ ] GPU available (8GB+ VRAM)
- [ ] CUDA 11.8+ installed
- [ ] Environment created (see NB 21 setup section)

## Section 2 — Pose Processing

Regardless of your detection method, you'll receive `rvec` and `tvec` (or a 4×4 matrix).
This section processes that pose into something actionable.

In [ ]:
# ── Section 2: Pose Processing ────────────────────────────────────────────

def pose_to_matrix(rvec, tvec):
    """Convert rvec + tvec to 4x4 homogeneous matrix."""
    R, _ = cv2.Rodrigues(rvec)
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = tvec.flatten()
    return T

def matrix_to_rvec_tvec(T):
    """Extract rvec + tvec from 4x4 homogeneous matrix."""
    R    = T[:3, :3]
    tvec = T[:3, 3]
    rvec, _ = cv2.Rodrigues(R)
    return rvec, tvec

def get_object_distance(tvec):
    """Euclidean distance from camera to object center."""
    return float(np.linalg.norm(tvec))

def get_euler_angles_deg(rvec):
    """
    Extract Euler angles (roll, pitch, yaw) from rvec.
    Returns degrees. Convention: ZYX (yaw, pitch, roll).
    """
    R, _ = cv2.Rodrigues(rvec)
    sy = np.sqrt(R[0, 0]**2 + R[1, 0]**2)
    if sy > 1e-6:
        roll  = np.degrees(np.arctan2(R[2, 1], R[2, 2]))
        pitch = np.degrees(np.arctan2(-R[2, 0], sy))
        yaw   = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    else:
        roll  = np.degrees(np.arctan2(-R[1, 2], R[1, 1]))
        pitch = np.degrees(np.arctan2(-R[2, 0], sy))
        yaw   = 0.0
    return roll, pitch, yaw

# ── YOUR POSE PROCESSING ──────────────────────────────────────────────────
# TODO: Add your custom processing below.
# Examples:
#   - Compute alignment errors (see NB 23)
#   - Compute grasp approach vector
#   - Compute AR overlay transform
#   - Compute relative pose from reference frame

def process_pose(object_id, rvec, tvec):
    """
    Your main pose processing function.
    rvec: (3,) or (3,1)
    tvec: (3,) or (3,1)
    Returns: whatever your application needs
    """
    dist    = get_object_distance(tvec.flatten())
    roll, pitch, yaw = get_euler_angles_deg(rvec)

    result = {
        'object_id': object_id,
        'distance_m': dist,
        'x_m': float(tvec.flatten()[0]),
        'y_m': float(tvec.flatten()[1]),
        'z_m': float(tvec.flatten()[2]),
        'roll_deg':  roll,
        'pitch_deg': pitch,
        'yaw_deg':   yaw,
        # TODO: add your custom fields
    }

    return result

# ── Test with dummy pose ───────────────────────────────────────────────────
rvec_test = np.array([0.2, 0.4, 0.1])
tvec_test = np.array([0.05, -0.03, 0.45])
result = process_pose(object_id=0, rvec=rvec_test, tvec=tvec_test)

print('Pose Processing Result:')
for k, v in result.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## Section 3 — Visualization

Build your HUD / overlay here. Reuse patterns from NB 14, 15, 23, 24.

In [ ]:
# ── Section 3: Visualization ──────────────────────────────────────────────

def draw_pose_overlay(frame: np.ndarray, rvec: np.ndarray, tvec: np.ndarray,
                      K: np.ndarray, dist: np.ndarray,
                      label: str = '', axis_length: float = 0.05) -> np.ndarray:
    """Draw coordinate axes + label at estimated pose."""
    out = frame.copy()
    try:
        cv2.drawFrameAxes(out, K, dist, rvec.astype(np.float32),
                          tvec.flatten().astype(np.float32), axis_length)
    except Exception:
        pass

    if label:
        # Project label to 2D
        origin_3d = np.array([[0.0, 0.0, 0.0]], dtype=np.float32)
        p2d, _ = cv2.projectPoints(origin_3d, rvec.astype(np.float32),
                                    tvec.flatten().astype(np.float32), K, dist)
        px, py = int(p2d[0][0][0]), int(p2d[0][0][1])
        h, w = frame.shape[:2]
        if 0 <= px < w and 0 <= py < h:
            cv2.putText(out, label, (px + 5, py - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return out

def draw_info_panel(frame: np.ndarray, pose_result: dict) -> np.ndarray:
    """Draw a clean information panel on the frame."""
    out = frame.copy()
    h, w = out.shape[:2]

    # Dark panel background
    panel_h = 120
    cv2.rectangle(out, (0, 0), (w, panel_h), (15, 15, 25), -1)

    cv2.putText(out, f'Object {pose_result["object_id"]}', (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(out,
                f'X={pose_result["x_m"]*100:+.1f}cm  '
                f'Y={pose_result["y_m"]*100:+.1f}cm  '
                f'Z={pose_result["z_m"]*100:.1f}cm',
                (10, 56), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 200, 255), 1)
    cv2.putText(out,
                f'Roll={pose_result["roll_deg"]:+.1f}°  '
                f'Pitch={pose_result["pitch_deg"]:+.1f}°  '
                f'Yaw={pose_result["yaw_deg"]:+.1f}°',
                (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 180, 80), 1)
    cv2.putText(out, f'Distance: {pose_result["distance_m"]:.3f} m',
                (10, 104), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 255, 100), 1)

    # TODO: Add your own fields to the panel
    return out

# ── Test visualization ────────────────────────────────────────────────────
test_frame = np.full((480, 640, 3), (30, 30, 40), dtype=np.uint8)
# Add fake object
cv2.rectangle(test_frame, (240, 170), (400, 310), (80, 100, 120), -1)
cv2.rectangle(test_frame, (240, 170), (400, 310), (140, 170, 200), 2)

result = process_pose(object_id=0, rvec=rvec_test, tvec=tvec_test)
out = draw_pose_overlay(test_frame, rvec_test, tvec_test, K, DIST, label='Object 0')
out = draw_info_panel(out, result)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title('Visualization Template — customize for your project', fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## Section 4 — Output / Action

Connect your pose estimate to the output. This is where the robotics magic happens.

In [ ]:
# ── Section 4: Output / Action ────────────────────────────────────────────

# Choose one or more of the following patterns:

# ── Pattern A: Print pose to terminal (simplest) ──────────────────────────
def output_to_terminal(pose_result):
    print(f'[{pose_result["object_id"]}] '
          f'Z={pose_result["z_m"]:.3f}m  '
          f'X={pose_result["x_m"]*100:+.1f}cm  '
          f'yaw={pose_result["yaw_deg"]:+.1f}deg')

# ── Pattern B: Send over ROS topic ───────────────────────────────────────
# import rospy
# from geometry_msgs.msg import PoseStamped
# def output_to_ros(pose_result):
#     msg = PoseStamped()
#     msg.header.stamp = rospy.Time.now()
#     msg.header.frame_id = 'camera_frame'
#     msg.pose.position.x = pose_result['x_m']
#     msg.pose.position.y = pose_result['y_m']
#     msg.pose.position.z = pose_result['z_m']
#     # Fill quaternion from rvec using transforms3d or scipy
#     pub.publish(msg)

# ── Pattern C: Send to robot via TCP socket ───────────────────────────────
# import socket, json
# def output_to_robot_socket(pose_result, host='192.168.1.100', port=9000):
#     payload = json.dumps(pose_result).encode()
#     with socket.socket() as s:
#         s.connect((host, port))
#         s.sendall(payload)

# ── Pattern D: Log to CSV file ────────────────────────────────────────────
import csv, datetime
def output_to_csv(pose_result, csv_path='pose_log.csv'):
    file_exists = Path(csv_path).exists()
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['timestamp'] + list(pose_result.keys()))
        if not file_exists:
            writer.writeheader()
        row = {'timestamp': datetime.datetime.now().isoformat(), **pose_result}
        writer.writerow(row)

# ── Test output ───────────────────────────────────────────────────────────
print('Testing output patterns...')
output_to_terminal(result)
output_to_csv(result)
print('CSV log written to pose_log.csv')

# TODO: Enable the output pattern that fits your project

## Section 5 — Main Loop

The full real-time pipeline. Start here once all other sections are working.

In [ ]:
# ── Section 5: Main Pipeline Loop ─────────────────────────────────────────

# Uncomment and run this cell to start the live pipeline.
# Press 'q' to quit, 's' to save a screenshot.

# cap = cv2.VideoCapture(CAMERA_INDEX)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH,  FRAME_WIDTH)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
#
# print(f'Starting {PROJECT_NAME}')
# print('Press q to quit, s to screenshot')
#
# frame_count = 0
# t_start = time.time()
#
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         print('Camera read failed'); break
#
#     frame_count += 1
#     fps = frame_count / (time.time() - t_start)
#
#     # ── DETECTION ──────────────────────────────────────────────────────────
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     # TODO: run your chosen detection method
#     # corners, ids, _ = detect_markers(gray, ARUCO_DICT, ARUCO_PARAMS)
#
#     # ── POSE ESTIMATION ────────────────────────────────────────────────────
#     # TODO: estimate pose from detections
#     # rvec, tvec = ...
#
#     # ── PROCESSING ────────────────────────────────────────────────────────
#     # result = process_pose(object_id, rvec, tvec)
#
#     # ── VISUALIZATION ─────────────────────────────────────────────────────
#     # frame = draw_pose_overlay(frame, rvec, tvec, K, DIST)
#     # frame = draw_info_panel(frame, result)
#     cv2.putText(frame, f'FPS: {fps:.1f}', (frame.shape[1]-120, 30),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.7, (100, 255, 100), 2)
#
#     # ── OUTPUT ────────────────────────────────────────────────────────────
#     # output_to_terminal(result)
#
#     # ── DISPLAY ───────────────────────────────────────────────────────────
#     cv2.imshow(PROJECT_NAME, frame)
#     key = cv2.waitKey(1) & 0xFF
#     if key == ord('q'):
#         break
#     elif key == ord('s'):
#         fname = f'screenshot_{frame_count:04d}.jpg'
#         cv2.imwrite(fname, frame)
#         print(f'Saved {fname}')
#
# cap.release()
# cv2.destroyAllWindows()
# print(f'Done. Ran for {time.time()-t_start:.1f}s, {frame_count} frames')

print('Main loop template ready.')
print('Uncomment the code above once your detection and processing are working.')

## Section 6 — Project Checklist & Grading Rubric

Use this to evaluate your own project before submitting.

### Part A — Core Implementation (60 pts)

| Item | Points | Done? |
|------|--------|-------|
| Camera calibrated (not using defaults) | 10 | ☐ |
| Chosen method justification written | 5 | ☐ |
| Detection working on real data | 15 | ☐ |
| Pose estimation producing valid rvec/tvec | 15 | ☐ |
| Visualization overlay renders correctly | 10 | ☐ |
| Output / action connected | 5 | ☐ |

### Part B — Quality (25 pts)

| Item | Points | Done? |
|------|--------|-------|
| Handles missing detections gracefully | 5 | ☐ |
| Code is clean and commented | 5 | ☐ |
| Quantitative accuracy measured (error in cm/deg) | 10 | ☐ |
| FPS measured and reported | 5 | ☐ |

### Part C — Extension (15 pts — choose one)

| Extension | Points | Done? |
|-----------|--------|-------|
| Multi-object tracking | 15 | ☐ |
| Kalman filter for smooth pose | 15 | ☐ |
| Occlusion handling with timeout | 15 | ☐ |
| ROS integration | 15 | ☐ |
| Full robot command loop | 15 | ☐ |

## Section 7 — Reflection & Next Steps

*(Fill this in after completing your project.)*

**What worked well?**

---

**What was harder than expected?**

---

**Measured accuracy:**
- Translation error: ___ cm (average)
- Rotation error: ___ degrees (average)
- FPS achieved: ___

---

**If you had 2 more weeks, what would you add?**

---

**Which notebook from this course was most useful?**

## Course Complete! — The Full Journey

You've covered the **complete 6D pose estimation stack**:

```
  NB 00–01  Getting started + environment
  NB 02–03  Jupyter + NumPy fundamentals
  NB 04–05  OpenCV foundations (images, color, transforms)
  NB 06–08  Camera model, calibration, distortion
  NB 09–10  solvePnP: classical 6D pose from 2D-3D correspondences
  NB 11–15  ArUco: generation, detection, pose estimation, robotics app
  NB 16–17  Stereo vision: disparity, depth, calibration
  NB 18     Deep learning for computer vision: CNNs, training, GPU
  NB 19     MediaPipe Objectron: 3D detection in real-time on CPU
  NB 20     EfficientPose: end-to-end 6D without PnP
  NB 21     FoundationPose + FreeZe: zero-shot 6D with foundation models
  NB 22     MegaPose + ViSP: CAD-based pose + real-time tracking
  NB 23     Full project: ArUco station alignment with state machine
  NB 24     Full project: Pallet detection + fork alignment
  NB 25     This capstone — YOUR project
```

You now have a complete mental model of **everything from pinhole camera geometry to foundation model zero-shot pose estimation**. That puts you in the top fraction of engineers working in robotics perception.

**What's next?**

- **ROS2**: Wrap your pipeline in a ROS2 node for real robot integration
- **Kalman/Particle filters**: Smooth noisy pose estimates over time
- **Grasp planning**: Use 6D pose to compute robot gripper transforms
- **Neural Radiance Fields (NeRF)**: The next frontier in 3D scene representation
- **BOP Benchmark**: Submit your method to the official 6D pose leaderboard

---

*Good luck with your project. You've got the tools.*

In [ ]:
# ── Course Completion Stats ───────────────────────────────────────────────

notebooks = [
    'NB 00: Welcome & Roadmap',
    'NB 01: Environment Setup',
    'NB 02: Jupyter 101',
    'NB 03: NumPy for CV',
    'NB 04: Intro to OpenCV',
    'NB 05: Image Operations',
    'NB 06: Camera Model Theory',
    'NB 07: Camera Calibration',
    'NB 08: Distortion & Undistortion',
    'NB 09: solvePnP Explained',
    'NB 10: Pose with Chessboard',
    'NB 11: ArUco Theory',
    'NB 12: Generate ArUco',
    'NB 13: Detect ArUco',
    'NB 14: ArUco Pose Estimation',
    'NB 15: ArUco Robotics App',
    'NB 16: Stereo Theory',
    'NB 17: Stereo Calibration',
    'NB 18: Intro to DL for CV',
    'NB 19: MediaPipe Objectron',
    'NB 20: EfficientPose',
    'NB 21: FoundationPose & FreeZe',
    'NB 22: MegaPose + ViSP',
    'NB 23: Station Alignment Project',
    'NB 24: Pallet Detection Project',
    'NB 25: Capstone ← YOU ARE HERE',
]

fig, ax = plt.subplots(figsize=(10, 12))
ax.set_facecolor('#0d0d1a')
fig.patch.set_facecolor('#0d0d1a')

part_colors = [
    '#95a5a6',  # NB 00-01
    '#95a5a6',
    '#1abc9c',  # NB 02-03
    '#1abc9c',
    '#3498db',  # NB 04-05
    '#3498db',
    '#9b59b6',  # NB 06-08
    '#9b59b6',
    '#9b59b6',
    '#e74c3c',  # NB 09-10
    '#e74c3c',
    '#e67e22',  # NB 11-15
    '#e67e22',
    '#e67e22',
    '#e67e22',
    '#e67e22',
    '#f1c40f',  # NB 16-17
    '#f1c40f',
    '#2ecc71',  # NB 18-22
    '#2ecc71',
    '#2ecc71',
    '#2ecc71',
    '#2ecc71',
    '#e74c3c',  # NB 23-25
    '#e74c3c',
    '#ffffff',  # Capstone
]

for i, (nb, color) in enumerate(zip(reversed(notebooks), reversed(part_colors))):
    y = i
    ax.barh(y, 1, color=color, alpha=0.85, height=0.7)
    ax.text(0.02, y, nb, va='center', ha='left', color='white',
            fontsize=9, fontfamily='monospace')

ax.set_xlim(0, 1.2)
ax.set_ylim(-0.5, len(notebooks) - 0.5)
ax.axis('off')
ax.set_title('6D Pose Vision Workshop — Complete!', color='white',
             fontsize=16, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print(f'\n  Total notebooks: {len(notebooks)}')
print(f'  Full stack covered: Camera model → Deep Learning → Zero-shot Foundation Models')
print(f'  Now go build something that moves.')